# UNET for statistical downscaling (RCM predictors → 3 km target)

This notebook is `UNET.py` split into its key parts, with extra comments. It is organised as:

1. **Imports**
2. **Basic building blocks** – dense layer, convolutional layer, convolutional block
3. **U-Net specific blocks** – contractive (encoder) block, upsampling layer, skip-connection concat, expansive (decoder) block
4. **Input / bottleneck / output blocks**
5. **The `UNET` model** – assembles everything and defines the forward pass
6. **Data configuration and loading** – reading NetCDF files, grid checks
7. **Pre-processing** – chronological split, normalisation, padding
8. **Downscaling wrapper, loss and training loop**
9. **Running the training** (replaces the `argparse` command line of the script)

**U-Net in one picture**

```
input ─► InputBlock2D ─► [Contractive 1] ─► [Contractive 2] ─► ... ─► [Contractive N]
              │                │                  │                         │
              │ skip           │ skip             │ skip                    ▼
              ▼                ▼                  ▼                    (Bottleneck, optional 1D input)
output ◄─ OutputBlock ◄─ [Expansive N] ◄─ ... ◄─ [Expansive 2] ◄─ [Expansive 1]
```
The encoder halves the spatial size and increases the channels; the decoder does the opposite, and at every
level it concatenates the encoder feature map of the same size (**skip connection**) to recover fine spatial detail.

## 1. Imports

In [ ]:
import argparse
import glob
import json
from pathlib import Path
from types import SimpleNamespace   # used in section 9 to replace argparse

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

## 2. Basic building blocks

Three small reusable pieces. Every other block of the network is built from them.

* `DenseLayer`: `Linear → BatchNorm1d → ReLU → Dropout` (used for the optional 1D input)
* `ConvolutionalLayer`: `Conv2d → BatchNorm2d → ReLU → Dropout2d`
* `ConvolutionalBlock`: two `ConvolutionalLayer` in a row (the classic "double 3x3 conv" of the U-Net paper)

In [ ]:
class DenseLayer(nn.Sequential):
    """Fully connected layer: Linear -> BatchNorm -> ReLU -> Dropout."""
    def __init__(self, in_features, out_features, dropout_rate = 0) -> None:
        super().__init__()

        self.block = nn.Sequential(
            nn.Linear(in_features = in_features, out_features = out_features),
            nn.BatchNorm1d(num_features = out_features),   # normalises each feature over the batch
            nn.ReLU(),
            nn.Dropout1d(p = dropout_rate)                 # p = 0 -> dropout disabled
            )

In [ ]:
class ConvolutionalLayer(nn.Sequential):
    """2D convolution -> BatchNorm -> ReLU -> Dropout. 'same' padding keeps H and W unchanged."""
    def __init__(self, in_channels, out_channels, kernel_size = 3, dropout_rate = 0,
                 padding = 'same', padding_mode = "zeros",
                 **kwargs) -> None:
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(kernel_size = kernel_size, padding = padding, padding_mode = padding_mode,
                      in_channels = in_channels, out_channels = out_channels,
                      bias = False,                        # BatchNorm already has its own bias (shift) term
                      **kwargs),
            nn.BatchNorm2d(num_features = out_channels),
            nn.ReLU(),
            nn.Dropout2d(p = dropout_rate)                 # drops whole channels (feature maps)
            )

In [ ]:
class ConvolutionalBlock(nn.Sequential):
    """Two ConvolutionalLayers. The first one changes the number of channels
    (in_channels -> in_channels * multiplies_channels_by), the second keeps it."""
    def __init__(self, in_channels, kernel_size = 3, multiplies_channels_by = 1, dropout_rate = 0,
                 padding = 'same', padding_mode = 'zeros') -> None:
        super().__init__()

        # int() because multiplies_channels_by can be 1/2 in the decoder
        self.out_channels = int(multiplies_channels_by*in_channels)

        self.block = nn.Sequential(
            ConvolutionalLayer(in_channels = in_channels, out_channels = self.out_channels, kernel_size = kernel_size,
                               dropout_rate = dropout_rate, padding = padding, padding_mode = padding_mode),
            ConvolutionalLayer(in_channels = self.out_channels, out_channels = self.out_channels, kernel_size = kernel_size,
                               dropout_rate = dropout_rate, padding = padding, padding_mode = padding_mode)
            # idea: add an optional residual connection here (would require a non-sequential module)
            )

## 3. U-Net specific blocks

### 3.1 Contractive (encoder) block
`MaxPool(2x2)` halves H and W, then a `ConvolutionalBlock` (usually) doubles the channels.

In [ ]:
class ContractiveBlock(nn.Sequential):
    """Encoder step: downsample by 2 (MaxPool) then ConvolutionalBlock."""
    def __init__(self, in_channels, kernel_size = 3, multiplies_channels_by = 2, dropout_rate = 0, padding = 'same', padding_mode = 'zeros') -> None:
        super().__init__()

        convolutional_block = ConvolutionalBlock(in_channels = in_channels,
                                                 dropout_rate = dropout_rate,
                                                 kernel_size = kernel_size,
                                                 multiplies_channels_by = multiplies_channels_by,
                                                 padding = padding,
                                                 padding_mode = padding_mode )

        self.block = nn.Sequential(
            nn.MaxPool2d(kernel_size = 2, stride = 2),     # (H, W) -> (H/2, W/2)
            convolutional_block
            )

        self.out_channels = convolutional_block.out_channels   # needed by the UNET to size the next block

### 3.2 Upsampling layer (doubles the resolution)
Two versions are kept:

* `DoubleResolutionLayer_old` – transposed convolution (2x2, stride 2). It can create *checkerboard* artefacts and is **not used**.
* `DoubleResolutionLayer` – **used**: bilinear upsampling followed by a normal 3x3 convolution, which avoids the checkerboard effect.

In [ ]:
class DoubleResolutionLayer_old(nn.Sequential):
    """OLD version (unused): transposed convolution. Kept for reference."""
    def __init__(self, in_channels, halves_channels = True, output_padding = (0,0),
                dropout_rate = 0) -> None:
        super().__init__()
        self.out_channels = in_channels

        if (halves_channels):
            self.out_channels = int(self.out_channels/2)

        # With ConvTranspose a 2x2 kernel is common; a 3x3 one creates the checkerboard effect.
        # output_padding lets us recover an odd original size (e.g. 25 -> 12 -> 24 (+1) -> 25).
        self.conv_transpose = nn.ConvTranspose2d(in_channels = in_channels,
                                                 out_channels = self.out_channels,
                                                 kernel_size = 2, stride = 2,
                                                 output_padding = output_padding)

In [ ]:
class DoubleResolutionLayer(nn.Module):
    """Upsample x2 (bilinear) then 3x3 convolution. Removes the checkerboard effect of transposed convolutions."""
    def __init__(self, in_channels, halves_channels = True, output_padding = (0,0), dropout_rate=0.0) -> None:
        super().__init__()
        # output_padding is accepted only to keep the same signature as the old layer; it is not used here

        out_channels = in_channels // 2 if halves_channels else in_channels
        self.out_channels = out_channels

        self.upsample = nn.Upsample(scale_factor = 2, mode = "bilinear")   # (H, W) -> (2H, 2W)
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=3,
                      padding=1, padding_mode = "replicate",               # replicate border values instead of zeros
                      bias=False),
            nn.BatchNorm2d(num_features=out_channels),
            nn.ReLU(),
            nn.Dropout2d(p=dropout_rate),
        )

    def forward(self, x):
        x = self.upsample(x)
        x = self.conv(x)
        return x

### 3.3 Skip-connection concatenation
Glues the upsampled decoder feature map (`x_1`) and the encoder feature map of the same size (`x_2`) along the channel axis.

In [ ]:
class ConcatChannelsLayer(nn.Module):
    def __init__(self, x1_channels, x2_channels) -> None:
        super().__init__()

        self.out_channels = x1_channels + x2_channels      # channels simply add up

    def forward(self, x_1, x_2):
        # tensors are (N, C, H, W): dim = 1 is the channel dimension
        x = torch.concat(tensors = (x_1, x_2), dim = 1)

        return(x)

### 3.4 Expansive (decoder) block
From the U-Net paper: *every step in the expansive path consists of an upsampling of the feature map, a convolution that
halves the number of channels, a concatenation with the corresponding feature map from the contracting path, and two 3x3
convolutions.* (Here we use padding `'same'`, so no cropping of the skip feature map is needed.)

Set `skip_x_out_channels = 0` for a block **without** skip connection (used in the `OutputBlock`).

In [ ]:
class ExpansiveBlock(nn.Module):
    """Decoder step: upsample x2 -> (concat skip) -> ConvolutionalBlock."""
    def __init__(self,
                 in_channels, skip_x_out_channels,
                 halves_channels_in_resolution_increase,
                 halves_channels_after_convolutional_block,
                 kernel_size = 3,
                 dropout_rate = 0, output_padding = (0,0),
                 ) -> None:
        super().__init__()

        self.skip_x_out_channels = skip_x_out_channels

        # 1) double H and W (and optionally halve the channels)
        self.double_resolution_layer = DoubleResolutionLayer(in_channels = in_channels,
                                                             halves_channels = halves_channels_in_resolution_increase,
                                                             output_padding = output_padding)

        # 2) channels entering the convolutional block = upsampled channels (+ skip channels, if any)
        if skip_x_out_channels > 0:
            self.concat_channels_layer = ConcatChannelsLayer(x1_channels = self.double_resolution_layer.out_channels,
                                                             x2_channels = skip_x_out_channels)
            in_channels = self.concat_channels_layer.out_channels
        else:
            in_channels = self.double_resolution_layer.out_channels

        # 3) two 3x3 convolutions, optionally halving the channels
        if halves_channels_after_convolutional_block:
            multiplies_channels_by = 1/2
        else:
            multiplies_channels_by = 1

        self.convolutional_block = ConvolutionalBlock(in_channels = in_channels, kernel_size = kernel_size,
                                                      multiplies_channels_by = multiplies_channels_by,
                                                      dropout_rate = dropout_rate)

        self.out_channels = self.convolutional_block.out_channels

    def forward(self, x, x_from_skip_connection = None):
        x = self.double_resolution_layer(x)

        if self.skip_x_out_channels > 0:
            x = self.concat_channels_layer(x, x_from_skip_connection)

        x = self.convolutional_block(x)

        return(x)

## 4. Bottleneck, input and output blocks

### 4.1 Bottleneck block (only used when a 1D input is provided)
The deepest 2D feature map is squeezed to 1x1 with a convolution, concatenated with the output of the 1D (dense) branch,
and expanded back to its original spatial shape with a transposed convolution. This is how scalar/vector predictors
(not on a grid) are injected into the U-Net.

In [ ]:
class BottleNeckBlock(nn.Sequential):
    """Fuses the deepest 2D feature map with the output of the 1D dense branch.
    dimensions_2D_input = (channels, H, W) -- does NOT include the batch dimension."""

    def __init__(self, dimensions_2D_input, input1Dblockoutput_features) -> None:
        super().__init__()

        kernel_size = dimensions_2D_input[1:3]      # (H, W) of the deepest feature map
        in_channels = dimensions_2D_input[0]

        # (N, F) -> (N, F, 1, 1) so it can be concatenated with a 1x1 feature map
        self.unflatten1D = nn.Unflatten(dim = 1, unflattened_size = (input1Dblockoutput_features, 1, 1) )

        if ( (kernel_size[0]) == 1 and (kernel_size[1] == 1) ):
            # already 1x1: nothing to squeeze / expand
            self.convolution_to_1_1 = nn.Identity()
            self.moveBacktoOriginal2Dshape = nn.Identity()
        else:
            # kernel = full (H, W) with no padding -> output is exactly 1x1
            self.convolution_to_1_1 = nn.Sequential(
                nn.Conv2d(kernel_size = kernel_size, padding = 0,
                          in_channels = in_channels, out_channels = in_channels,
                          bias = False),  # already a bias term in batchnorm
                nn.BatchNorm2d(num_features = in_channels)
                )

            # 1x1 -> (H, W) again
            self.moveBacktoOriginal2Dshape = nn.ConvTranspose2d(in_channels = in_channels + input1Dblockoutput_features,
                                                                out_channels = in_channels + input1Dblockoutput_features,
                                                                kernel_size = kernel_size,
                                                                stride = 1)

        self.out_channels = in_channels + input1Dblockoutput_features

    def forward(self, x, input1Dblock_output):
        x = self.convolution_to_1_1(x)                      # (N, C, H, W) -> (N, C, 1, 1)
        x1d = self.unflatten1D(input1Dblock_output)         # (N, F)       -> (N, F, 1, 1)

        xx = torch.concat(tensors = (x1d, x), dim = 1)      # (N, F + C, 1, 1)

        xx = self.moveBacktoOriginal2Dshape(xx)             # back to (N, F + C, H, W)

        return( xx )

### 4.2 Input blocks

In [ ]:
class InputBlock2D(nn.Sequential):
    """First processing of the gridded predictors: two ConvolutionalLayers -> number_of_kernels channels."""
    def __init__(self, input_shape, reshaped_input_size = None, kernel_size = 3, number_of_kernels = 64, dropout_rate = 0,
                 padding = 'same', padding_mode = 'zeros') -> None:
        super().__init__()

        self.block = nn.Sequential(
            # input_shape = (N, C, H, W) -> input_shape[1] is the number of predictor channels
            ConvolutionalLayer(in_channels = input_shape[1], out_channels = number_of_kernels, kernel_size = kernel_size,
                               dropout_rate = dropout_rate, padding = padding, padding_mode = padding_mode),
            ConvolutionalLayer(in_channels = number_of_kernels, out_channels = number_of_kernels, kernel_size = kernel_size,
                               dropout_rate = dropout_rate, padding = padding, padding_mode = padding_mode)
            )

        self.out_channels = number_of_kernels

In [ ]:
class InputBlock1D(nn.Sequential):
    """MLP for the optional non-gridded input: a stack of DenseLayers, one per entry of neurons_dense_list."""
    def __init__(self, input_shape, neurons_dense_list, dropout_rate = 0) -> None:
        super().__init__()

        self.block = nn.Sequential()

        in_features = input_shape[1]                        # input_shape = (N, F)
        for features in neurons_dense_list:
            self.block.append( DenseLayer(in_features = in_features, out_features = features,
                                          dropout_rate = dropout_rate) )
            in_features = features

        self.out_features = features                        # size of the last dense layer (feeds the bottleneck)

### 4.3 Output block
Optionally increases the resolution `double_resolution_times` times (each time x2, no skip connection) and ends with a
1x1 convolution that maps to the requested number of output channels. `out_channels` can be:

* an **int** – number of output channels, no activation (our case: `1`, the temperature)
* a **list of activation classes** – one output channel per activation (e.g. `[nn.Softplus, nn.Sigmoid]`)

In [ ]:
class OutputBlock(nn.Module):
    # Inherits from Module (not Sequential) because ExpansiveBlock.forward is not a plain single-input forward.
    def __init__(self,  in_channels, out_channels,
                 kernel_size = 3, double_resolution_times = 0, dropout_rate = 0) -> None:
        super().__init__()

        self.output_block = nn.ModuleList()

        # Extra upsampling stages (e.g. 2 stages = x4: 12 km -> 3 km). None if the input is already on the target grid.
        if (double_resolution_times > 0):
            for i in range(double_resolution_times):
                if ( i == (double_resolution_times -1) ):
                    dropout_rate = 0                        # no dropout right before the final 1x1 conv

                block = ExpansiveBlock(in_channels = in_channels,
                                       skip_x_out_channels = 0,                          # no skip available at these scales
                                       kernel_size = kernel_size,
                                       halves_channels_after_convolutional_block = False,
                                       halves_channels_in_resolution_increase = False,   # keep the channels constant
                                       dropout_rate = dropout_rate)
                in_channels = block.out_channels
                self.output_block.append( block )

        # Per-channel output activations (if out_channels is a list of activation classes):
        if type(out_channels) is list:
            self.output_activations_layer = nn.ModuleList()
            for out_activation in out_channels:
                self.output_activations_layer.append( out_activation() )

            self.out_channels = len(out_channels)
        else:
            self.output_activations_layer = None
            self.out_channels = out_channels

        # Final 1x1 convolution: feature channels -> output variables
        self.output_block.append(
                    nn.Conv2d(in_channels = in_channels, out_channels = self.out_channels,
                              kernel_size = 1, padding = 0)
                                )

    def forward(self, x):
        for layer in self.output_block:
            x = layer(x)

        if (self.output_activations_layer is None):
            return(x)

        else:
            # apply activation i to output channel i, then re-stack the channels
            xx = self.output_activations_layer[0]( x[:,[0],:,:] )
            for i in range(1, x.shape[1]):                  # output of conv2d is (N, C, H, W)
                xx = torch.cat( (xx, self.output_activations_layer[i]( x[:,[i], ...])), dim = 1)

            return(xx)

## 5. The `UNET` model

The constructor does four things:

1. **Handles optional inputs** – a custom 2D pre-processing block and/or a 1D input.
2. **Computes `out_padding_pattern`** – tracks which levels have odd H/W (kept from the transposed-conv version).
3. **Builds the contractive path**, recording the channels of each skip connection.
4. **Builds the (optional) bottleneck, the expansive path (in reverse order) and the output block.**

`forward` stores a copy of the feature map before each contractive block (the skip connections), then consumes them
in reverse order in the expansive path.

In [ ]:
class UNET(nn.Module):

    def __init__(self,
                 input_shape_2D, output_channels,
                 input_shape_1D = None,
                 input_processing_block_2D = None,
                 input_number_kernels = 64,
                 number_of_blocks = 4,
                 inputNN_neurons_list = [64, 128, 256, 512],
                 kernel_size = 3,
                 double_resolution_times = 2,
                 dropout_rate_1Dinput = 0.,
                 dropout_rate_2D = 0.,
                 debug_sizes = False) -> None:
        # input_shape_2D: (N, Variables, Height, Width)
        # input_processing_block_2D: optional block that processes the 2D input first (e.g. cropping);
        #     it must have an `output_shape` attribute in the same format as input_shape_2D
        # double_resolution_times: extra x2 upsampling stages at the output (scale_factor = 2 ** double_resolution_times)
        super().__init__()

        self.debug_sizes = debug_sizes
        self.number_of_blocks = number_of_blocks

        # ---- 1) optional input processing ----------------------------------------------------------
        if not (input_processing_block_2D is None):
            self.input_processing_block_2D = input_processing_block_2D
            self.input_shape_2D = self.input_processing_block_2D.output_shape
        else:
            self.input_processing_block_2D = None
            self.input_shape_2D = input_shape_2D

        self.input_shape_1D = input_shape_1D

        # ---- 2) odd-size tracker --------------------------------------------------------------------
        # For each level: 1 if (H, W) is odd (needs output_padding to be restored after halving), else 0.
        self.out_padding_pattern = torch.zeros(number_of_blocks, 2)
        dimension_tracker = torch.tensor(self.input_shape_2D[2:4])      # (H, W)

        for i in range(number_of_blocks):
            self.out_padding_pattern[i , :] = (dimension_tracker % 2) != 0
            dimension_tracker = (dimension_tracker - (dimension_tracker % 2)) / 2   # floor(size / 2), like MaxPool
        # after the loop dimension_tracker = (H, W) of the deepest feature map

        self.out_padding_pattern = self.out_padding_pattern.type(torch.int32)

        # ---- The model --------------------------------------------------------------------------------
        self.input_block2D = InputBlock2D(input_shape = self.input_shape_2D, kernel_size = kernel_size,
                                          number_of_kernels = input_number_kernels,
                                          dropout_rate = dropout_rate_2D,
                                          padding_mode = "replicate")
        if self.input_shape_1D != None:
            self.input_block1D = InputBlock1D(input_shape = self.input_shape_1D, neurons_dense_list = inputNN_neurons_list,
                                          dropout_rate = dropout_rate_1Dinput)
        else:
            self.input_block1D = None

        # ---- 3) Contractive path (encoder) ----------------------------------------------------------
        in_channels = input_number_kernels
        self.contractivePath = nn.ModuleList()

        # channels of the feature maps that will be used as skip connections (the first one is the input block output)
        skip_x_out_channels_list = [ self.input_block2D.out_channels ]

        for i in range(number_of_blocks):
            if i == 0:
                block = ContractiveBlock( in_channels, kernel_size = kernel_size, dropout_rate = dropout_rate_2D,
                                         multiplies_channels_by = (2 if self.input_shape_1D == None else 1),
                                         padding_mode = "replicate")
            # NOTE: this is `if`, not `elif`: for i == 0 (when number_of_blocks > 1) the `else` below runs as well
            # and overwrites the block created above. Left unchanged to keep the behaviour of UNET.py.
            if i == (number_of_blocks-1):
                block = ContractiveBlock( in_channels, kernel_size = kernel_size, dropout_rate = dropout_rate_2D,
                                         multiplies_channels_by = (2 if self.input_shape_1D == None else 1) )
                # The last block has NO skip connection. In the original model it does not double the features
                # because the bottleneck adds them; with no 1D input (None) the channels are doubled instead.
            else:
                block = ContractiveBlock(in_channels, kernel_size = kernel_size, dropout_rate = dropout_rate_2D, multiplies_channels_by = 2)
                skip_x_out_channels_list.append(block.out_channels)

            in_channels = block.out_channels
            self.contractivePath.append( block )

        # ---- Bottleneck (only with 1D input) --------------------------------------------------------
        if self.input_shape_1D != None:
            self.bottleneck_block = BottleNeckBlock(dimensions_2D_input = (in_channels,
                                                                           int(dimension_tracker[0].item()),
                                                                           int(dimension_tracker[1].item())),
                                                                           input1Dblockoutput_features = self.input_block1D.out_features)
            in_channels = self.bottleneck_block.out_channels

        # ---- 4) Expansive path (decoder) ------------------------------------------------------------
        self.expansivePath = nn.ModuleList()

        for i in range(number_of_blocks):
            # skips are consumed in reverse order (deepest first), hence pop()
            skip_x_out_channels = skip_x_out_channels_list.pop()

            expansive_block = ExpansiveBlock(in_channels = in_channels,
                                             skip_x_out_channels = skip_x_out_channels,
                                             kernel_size = kernel_size,
                                             halves_channels_in_resolution_increase = True,
                                             halves_channels_after_convolutional_block = True,
                                             output_padding = self.out_padding_pattern[number_of_blocks - (i+1), ].tolist(),
                                             dropout_rate = dropout_rate_2D)

            self.expansivePath.append(expansive_block)

            in_channels = expansive_block.out_channels

        # ---- Output block ---------------------------------------------------------------------------
        self.output_block = OutputBlock(in_channels = in_channels, out_channels = output_channels,
                                        kernel_size = kernel_size,
                                        double_resolution_times = double_resolution_times,
                                        dropout_rate = dropout_rate_2D)

    def forward(self, x, x_1D = None):
        # shape comments assume no input_processing_block_2D; uncomment the prints to debug sizes

        if not (self.input_processing_block_2D is None):
            x = self.input_processing_block_2D(x)

        x = self.input_block2D(x)                           # (N, C_in, H, W) -> (N, base, H, W)

        if self.input_block1D != None:
            x_1D = self.input_block1D(x_1D)                 # (N, F) -> (N, out_features)

        # Encoder: save the feature map BEFORE each downsampling (clone so later in-place ops cannot modify it)
        skip_x_list = []
        for i in range(self.number_of_blocks):
            skip_x_list.append(x.clone())
            x = self.contractivePath[i](x)                  # H, W halved at every block

        if self.input_block1D != None:
            x = self.bottleneck_block(x = x, input1Dblock_output = x_1D)

        # Decoder: pair each expansive block with the skip of the same resolution (deepest skip first)
        for expasive_block, skip_x in zip(self.expansivePath, skip_x_list[::-1]):
            x = expasive_block(x = x, x_from_skip_connection = skip_x)

        x = self.output_block(x)                            # extra upsampling (if any) + 1x1 conv
        return(x)

## 6. Data configuration and loading

Task: predict a **3 km** daily field (`tas`) from **RCM predictors** (ALADIN63).

* Default (`SCALE_FACTOR = 1`): predictors have **already been upscaled onto the 3 km grid**, so input and output have the same shape and the `OutputBlock` does not upsample.
* For raw ~12 km predictors use `SCALE_FACTOR = 4` (→ `double_resolution_times = 2`).

Adjust the paths/patterns below to the NAS layout. `{var}` is replaced by each variable name.

In [ ]:
DATA_DIR = Path("/mnt/nas/Data")

# Predictors (upscaled to 3 km) and target (3 km). Adjust the patterns to the NAS layout.
LR_VARIABLES = ["tasAdjust", "hussAdjust", "hursAdjust", "prAdjust",
                "rsdsAdjust", "sfcWindAdjust", "tasmaxAdjust", "tasminAdjust"]
LR_PATTERN = str(DATA_DIR / "ALADIN63_upscaled_3km" / "daily" / "{var}" / "*.nc")   # placeholder: set to the upscaled files
HR_TARGET = "tas"
HR_PATTERN = str(DATA_DIR / "3km" / "{var}" / "*.nc")

OROG_PATH = None          # optional static 3 km orography file, added as an extra input channel
SCALE_FACTOR = 1          # 1 = same grid (upscaled predictors), 4 = raw 12 km -> 3 km
MODELS_DIR = Path("models") / "unet_3km"    # where weights / stats / splits are saved

In [ ]:
def load_variable(pattern, var):
    """Open all NetCDF files of one variable as a single DataArray with dims (time, y, x)."""
    import xarray as xr

    files = sorted(glob.glob(pattern.format(var = var)))
    if not files:
        raise FileNotFoundError(f"No files for '{var}' matching {pattern.format(var = var)}")

    ds = xr.open_mfdataset(files, combine = "by_coords")            # lazy, concatenated along time
    da = ds[var] if var in ds.data_vars else ds[list(ds.data_vars)[0]]   # fall back to the first variable in the file
    da = da.squeeze(drop = True)                                    # drop length-1 dimensions
    da["time"] = da["time"].dt.floor("D")                           # daily data: align on dates only (drop hours)
    return da.transpose("time", ...)                                # (time, y, x)


def load_orography(path):
    """Static orography field (y, x), used as an additional input channel."""
    import xarray as xr

    ds = xr.open_dataset(path)
    da = ds["orog"] if "orog" in ds.data_vars else ds[list(ds.data_vars)[0]]
    da = da.squeeze(drop = True)
    if "time" in da.dims:
        da = da.isel(time = 0, drop = True)
    return da                                                       # (y, x)

In [ ]:
def check_same_grid(reference, others):
    """Same-grid training only makes sense if every field sits on exactly the target pixels."""
    spatial_dims = [d for d in reference.dims if d != "time"]
    for da in others:
        da_dims = [d for d in da.dims if d != "time"]
        # 1) same (H, W)
        if da.shape[-2:] != reference.shape[-2:]:
            raise ValueError(f"'{da.name}' grid {da.shape[-2:]} != target grid {reference.shape[-2:]}")
        # 2) same coordinate values (tolerance 1e-4)
        for ref_name in list(spatial_dims) + ["lat", "lon"]:
            if ref_name in reference.coords and ref_name in da.coords:
                if not np.allclose(reference[ref_name].values, da[ref_name].values, atol = 1e-4, equal_nan = True):
                    raise ValueError(f"'{da.name}' coordinate '{ref_name}' differs from the target grid")
        # 3) different dimension names are only a warning (shapes already match)
        if da_dims != spatial_dims:
            print(f"Warning: '{da.name}' dims {da_dims} differ from target dims {spatial_dims}, shapes match")

In [ ]:
def load_data(lr_pattern, lr_variables, hr_pattern, hr_target, same_grid = True, orog_path = None):
    """Return X (T, C, h, w) = stacked predictors and Y (T, 1, H, W) = target, on their common days."""
    import xarray as xr

    lr = [load_variable(lr_pattern, v) for v in lr_variables]
    hr = load_variable(hr_pattern, hr_target)
    # keep only days present in ALL variables (inner join on time; spatial dims are excluded from alignment)
    *lr, hr = xr.align(*lr, hr, join = "inner", exclude = [d for d in hr.dims if d != "time"])

    static = [load_orography(orog_path)] if orog_path else []
    if same_grid:
        check_same_grid(hr, lr + static)
    elif static and static[0].shape != lr[0].shape[-2:]:
        raise ValueError(f"Orography grid {static[0].shape} != predictor grid {lr[0].shape[-2:]}")

    # orography has no time axis: broadcast it to (T, h, w) so it behaves as one more predictor channel
    channels = [da.values for da in lr] + [np.broadcast_to(da.values, lr[0].shape) for da in static]
    X = np.stack(channels, axis = 1).astype(np.float32)                  # (T, C, h, w)
    Y = hr.values[:, None].astype(np.float32)                            # (T, 1, H, W)
    print(f"Common days: {X.shape[0]} | LR {X.shape[2:]} -> HR {Y.shape[2:]}")
    return X, Y

## 7. Pre-processing

* **Chronological split** (train → val → test, no shuffling across time, to avoid leakage between neighbouring days)
* **Normalisation** with statistics computed on the **training set only**
* **Padding** so H and W are divisible by `2 ** number_of_blocks` (each encoder block halves them)

In [ ]:
def chronological_splits(n, val_frac = 0.15, test_frac = 0.15):
    """First 70 % of the days = train, next 15 % = validation, last 15 % = test."""
    n_test, n_val = int(n * test_frac), int(n * val_frac)
    idx = np.arange(n)
    return idx[:n - n_val - n_test], idx[n - n_val - n_test:n - n_test], idx[n - n_test:]


def normalization_stats(X, Y):
    """Per-channel mean/std for the predictors, single mean/std for the target. NaNs (sea) are ignored."""
    return {"x_mean": np.nanmean(X, axis = (0, 2, 3)).tolist(), "x_std": np.nanstd(X, axis = (0, 2, 3)).tolist(),
            "y_mean": float(np.nanmean(Y)), "y_std": float(np.nanstd(Y))}


def normalize(X, Y, stats):
    """z-score normalisation using the saved stats (1e-8 avoids division by zero)."""
    x_mean = np.array(stats["x_mean"], dtype = np.float32)[None, :, None, None]   # broadcast over (T, C, H, W)
    x_std = np.array(stats["x_std"], dtype = np.float32)[None, :, None, None] + 1e-8
    X = np.nan_to_num((X - x_mean) / x_std)                  # NaN (e.g. sea) -> 0 in the inputs
    Y = (Y - stats["y_mean"]) / (stats["y_std"] + 1e-8)       # NaN kept in target, masked in the loss
    return X, Y


def pad_to_multiple(x, multiple):
    """Pad bottom/right (replicating the border) so H and W are multiples of `multiple`.
    Returns the padded tensor and the original (h, w) to crop back afterwards."""
    h, w = x.shape[-2:]
    pad_h, pad_w = (-h) % multiple, (-w) % multiple
    return F.pad(x, (0, pad_w, 0, pad_h), mode = "replicate"), (h, w)

## 8. Downscaling wrapper, loss and training loop

* `DownscalingModel`: pads the input → runs the `UNET` → crops the padding → resizes to the exact target grid if needed.
* `masked_mse`: MSE only over valid (non-NaN) target pixels, so sea/missing pixels do not contribute.
* `run_epoch`: one pass over a loader; trains if an optimizer is given, otherwise evaluates.

In [ ]:
class DownscalingModel(nn.Module):
    """Wraps the UNET: pads the input, crops the (x scale_factor) output and resizes it onto the exact 3 km grid.
    With scale_factor = 1 (upscaled predictors) the UNET keeps the input resolution."""
    def __init__(self, in_channels, lr_shape, hr_shape, number_of_blocks = 4, scale_factor = SCALE_FACTOR, **unet_kwargs) -> None:
        super().__init__()

        self.multiple = 2 ** number_of_blocks                       # H, W must be divisible by this
        self.scale_factor = scale_factor
        self.hr_shape = tuple(hr_shape)
        padded = [s + (-s) % self.multiple for s in lr_shape]       # padded (H, W) seen by the UNET

        self.unet = UNET(input_shape_2D = (1, in_channels, *padded), output_channels = 1,
                         number_of_blocks = number_of_blocks,
                         double_resolution_times = int(np.log2(scale_factor)),   # 4 -> 2 stages of x2
                         **unet_kwargs)

    def forward(self, x):
        x, (h, w) = pad_to_multiple(x, self.multiple)
        x = self.unet(x)[..., :h * self.scale_factor, :w * self.scale_factor]    # remove the padding
        if x.shape[-2:] != self.hr_shape:                           # e.g. 12 km x4 doesn't exactly match the 3 km grid
            x = F.interpolate(x, size = self.hr_shape, mode = "bilinear", align_corners = False)
        return x

In [ ]:
def masked_mse(y_pred, y_true):
    """MSE ignoring NaN pixels of the target."""
    mask = ~torch.isnan(y_true)
    return ((y_pred[mask] - y_true[mask]) ** 2).mean()


def run_epoch(model, loader, device, optimizer = None):
    """One epoch. Training mode if an optimizer is passed, evaluation (no gradients) otherwise.
    Returns the sample-weighted mean loss."""
    model.train(optimizer is not None)                  # enables dropout / batchnorm updates only when training
    total, n = 0.0, 0
    with torch.set_grad_enabled(optimizer is not None):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            loss = masked_mse(model(x), y)
            if optimizer is not None:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total += loss.item() * x.shape[0]           # weight by batch size (last batch may be smaller)
            n += x.shape[0]
    return total / n

### Training procedure (`train`)
1. Load data and make the chronological split (indices saved in `splits.npz`)
2. Compute normalisation stats **on the training set only** and save them (needed at inference)
3. Build the `DataLoader`s and the model, Adam optimizer and `ReduceLROnPlateau` scheduler (halves the LR after 7 epochs without improvement)
4. Loop over epochs, save the best weights (lowest validation MSE), **early stopping** after `patience` epochs without improvement
5. Reload the best weights and report the **test RMSE in physical units** (`sqrt(MSE) * y_std`)

In [ ]:
def train(args):
    from torch.utils.data import DataLoader, TensorDataset

    MODELS_DIR.mkdir(parents = True, exist_ok = True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1) data + split
    X, Y = load_data(args.lr_pattern, args.lr_variables, args.hr_pattern, args.hr_target,
                     same_grid = args.scale_factor == 1, orog_path = args.orog_path)
    idx_train, idx_val, idx_test = chronological_splits(X.shape[0])
    np.savez(MODELS_DIR / "splits.npz", train = idx_train, val = idx_val, test = idx_test)

    # 2) normalisation (stats from the training days only -> no information leakage from val/test)
    stats = normalization_stats(X[idx_train], Y[idx_train])
    stats.update({"lr_variables": args.lr_variables, "hr_target": args.hr_target,
                  "orog_path": args.orog_path, "scale_factor": args.scale_factor})
    json.dump(stats, open(MODELS_DIR / "normalization_stats.json", "w"), indent = 2)
    X, Y = normalize(X, Y, stats)

    # 3) loaders, model, optimizer, scheduler
    def loader(idx, shuffle):
        return DataLoader(TensorDataset(torch.from_numpy(X[idx]), torch.from_numpy(Y[idx])),
                          batch_size = args.batch_size, shuffle = shuffle)
    train_loader, val_loader = loader(idx_train, True), loader(idx_val, False)

    model = DownscalingModel(in_channels = X.shape[1], lr_shape = X.shape[2:], hr_shape = Y.shape[2:],
                             scale_factor = args.scale_factor,
                             input_number_kernels = args.base_channels,
                             dropout_rate_2D = args.dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr = args.lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor = 0.5, patience = 7)
    print(f"Device: {device} | parameters: {sum(p.numel() for p in model.parameters()):,}")

    # 4) training loop with best-checkpoint saving and early stopping
    best_val, epochs_without_improvement = float("inf"), 0
    for epoch in range(1, args.epochs + 1):
        train_loss = run_epoch(model, train_loader, device, optimizer)
        val_loss = run_epoch(model, val_loader, device)
        scheduler.step(val_loss)
        print(f"Epoch {epoch:3d} | train MSE {train_loss:.4f} | val MSE {val_loss:.4f}")

        if val_loss < best_val:
            best_val, epochs_without_improvement = val_loss, 0
            torch.save(model.state_dict(), MODELS_DIR / "best.pt")
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= args.patience:
                print("Early stopping")
                break

    # 5) final evaluation on the held-out test days with the best weights
    model.load_state_dict(torch.load(MODELS_DIR / "best.pt", map_location = device))
    test_loss = run_epoch(model, loader(idx_test, False), device)
    rmse = np.sqrt(test_loss) * stats["y_std"]              # back to target units (e.g. K or degC)
    print(f"Best val MSE {best_val:.4f} | test RMSE {rmse:.4f} (target units) | weights: {MODELS_DIR / 'best.pt'}")

## 9. Run the training

In the script these values come from the command line (`argparse`). In the notebook, edit the `SimpleNamespace` below
instead. The defaults are the same as in `UNET.py`.

Quick sanity check first: the next cell builds the model on random data to verify shapes **without needing the NAS data**.

In [ ]:
# Sanity check (no data needed): 8 predictors on a 50 x 60 grid, same-grid output
model = DownscalingModel(in_channels = 8, lr_shape = (50, 60), hr_shape = (50, 60), scale_factor = 1, input_number_kernels = 8)
model.eval()
with torch.no_grad():
    out = model(torch.randn(2, 8, 50, 60))
print("output shape:", tuple(out.shape), "| parameters:", f"{sum(p.numel() for p in model.parameters()):,}")

In [ ]:
args = SimpleNamespace(
    lr_pattern = LR_PATTERN,         # glob of (upscaled) predictor files, '{var}' = variable name
    lr_variables = LR_VARIABLES,
    hr_pattern = HR_PATTERN,         # glob of 3 km target files
    hr_target = HR_TARGET,
    orog_path = OROG_PATH,           # optional orography file, added as an input channel
    scale_factor = SCALE_FACTOR,     # 1 = predictors already on the 3 km grid, 4 = raw 12 km predictors
    base_channels = 32,              # channels of the first conv layers (doubles at each encoder level)
    dropout = 0.,
    batch_size = 16,
    lr = 1e-4,
    epochs = 75,
    patience = 15,                   # early-stopping patience (epochs)
)

train(args)